# Advanced Data Analysis: Natural Language Processing (NLP)
## Sentiment Classification of Customer Reviews with TF-IDF

**Technique:** turn free text into numbers with **TF-IDF** (unigrams and bigrams) and train supervised classifiers to predict the sentiment of a review.

**Application:** businesses receive thousands of reviews and support messages. Automatic sentiment analysis flags unhappy customers early and shows which words and topics signal dissatisfaction.

**Workflow:** load data, check and de-duplicate, clean text, split, vectorise, compare three models with cross-validation, evaluate on a held-out test set, extract the words that drive each prediction, and write the project report.

**Data:** by default the notebook downloads real movie reviews (IMDB, 50,000 labelled reviews; a balanced 6,000-review sample is used). If the download is not possible, it falls back to the bundled `customer_reviews_nlp.csv` and says so. The report states which dataset was used.

## 1. Setup

In [ ]:
import io
import os
import re
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
BLUE, RED, GREY = "#2a7fbf", "#d6432f", "#888888"

# Works inside the repo (notebooks/ folder), in a cloned repo, and in Colab
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DATA_DIR, IMG_DIR, REPORT_DIR = ROOT / "data", ROOT / "images", ROOT / "reports"
for d in (IMG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)


def save_fig(name):
    plt.tight_layout()
    plt.savefig(IMG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


R = {}   # key results, used to write the report at the end

## 2. Load the data
`USE_REAL_DATA = True` downloads real IMDB reviews (needs internet; in Colab it works out of the box). Set it to `False` to use the bundled file instead.

In [ ]:
USE_REAL_DATA = True
N_PER_CLASS = 3000                     # reviews per class taken from IMDB (keeps training fast)
CSV_NAME = "customer_reviews_nlp.csv"

# Fallback for Colab: download the bundled CSV from your GitHub repo.
# Replace YOUR-REPO-NAME with your repository name (the branch is usually 'main').
GITHUB_USER, GITHUB_REPO, GITHUB_BRANCH = "sarkarsoniya8-cmyk", "YOUR-REPO-NAME", "main"
RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/data/{CSV_NAME}"


def load_imdb():
    """Balanced sample of real IMDB movie reviews (HuggingFace `datasets`)."""
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "15", "datasets"],
                       check=True, timeout=300)
        from datasets import load_dataset
    raw = load_dataset("imdb", split="train").to_pandas()
    raw = raw.groupby("label").sample(n=N_PER_CLASS, random_state=RANDOM_STATE)
    return pd.DataFrame({"review_text": raw["text"].values,
                         "sentiment": raw["label"].map({0: "Negative", 1: "Positive"}).values})


def load_bundled():
    """The bundled synthetic file: local folders first, then GitHub, then a Colab upload."""
    for p in (DATA_DIR / CSV_NAME, Path("data") / CSV_NAME, Path(CSV_NAME)):
        if p.exists():
            return pd.read_csv(p)
    try:
        return pd.read_csv(RAW_URL)
    except Exception:
        try:
            from google.colab import files
            print("Please choose customer_reviews_nlp.csv from your computer:")
            up = files.upload()
            return pd.read_csv(io.BytesIO(up[next(iter(up))]))
        except ImportError:
            raise FileNotFoundError(f"Could not find {CSV_NAME}. Put it in the repo data/ folder or set GITHUB_REPO above.")


df, meta = None, {}
if USE_REAL_DATA:
    try:
        df = load_imdb()
        meta = dict(name="IMDB Large Movie Review Dataset", synthetic=False,
                    desc=f"A balanced sample of {N_PER_CLASS * 2:,} real movie reviews (from the 25,000-review training split of the 50,000-review IMDB dataset), labelled Positive or Negative.")
        print("Loaded real IMDB reviews:", df.shape)
    except Exception as exc:
        print(f"Could not load the real dataset ({type(exc).__name__}: {exc}).\nFalling back to the bundled file.")
if df is None:
    df = load_bundled()
    meta = dict(name="Bundled customer reviews (synthetic)", synthetic=True,
                desc="The bundled `customer_reviews_nlp.csv`: telecom-style customer reviews labelled Positive, Neutral or Negative. The text is generated from a small set of template sentences, so it is much easier than real reviews.")
    print("Loaded the bundled file:", df.shape)
R["meta"] = meta
df.head()

## 3. Quality checks, de-duplication and text cleaning
Identical reviews must not appear in both the training and the test set, otherwise the model is tested on text it has already seen and the accuracy is inflated. We therefore remove duplicate texts **before** splitting.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)      # HTML tags such as <br />
    text = re.sub(r"[^a-z\s]", " ", text)     # punctuation and digits
    return re.sub(r"\s+", " ", text).strip()


df = df.dropna(subset=["review_text", "sentiment"]).copy()
df["clean_text"] = df["review_text"].map(clean_text)
n_before = len(df)
n_dup = int(df.duplicated("clean_text").sum())
df = df.drop_duplicates("clean_text").reset_index(drop=True)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
df["word_count"] = df["clean_text"].str.split().str.len()
print(f"Reviews before: {n_before:,} | duplicate texts removed: {n_dup:,} | reviews kept: {len(df):,}")
print(df["sentiment"].value_counts().to_string())
R.update(n_before=n_before, n_dup=n_dup, n_kept=len(df), class_counts=df["sentiment"].value_counts())
df[["review_text", "clean_text", "sentiment", "word_count"]].head()

In [ ]:
classes = sorted(df["sentiment"].unique())
palette = {"Negative": RED, "Neutral": GREY, "Positive": BLUE}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
counts = df["sentiment"].value_counts().reindex(classes)
bars = ax[0].bar(counts.index, counts.values, color=[palette.get(c, BLUE) for c in counts.index])
ax[0].bar_label(bars, padding=3)
ax[0].set_xlabel("Sentiment")
ax[0].set_ylabel("Number of reviews")
ax[0].set_title("Class distribution (after removing duplicates)")
bp = ax[1].boxplot([df.loc[df["sentiment"] == c, "word_count"] for c in classes], patch_artist=True)
for patch, c in zip(bp["boxes"], classes):
    patch.set_facecolor(palette.get(c, BLUE))
    patch.set_alpha(0.6)
ax[1].set_xticks(range(1, len(classes) + 1))
ax[1].set_xticklabels(classes)
ax[1].set_xlabel("Sentiment")
ax[1].set_ylabel("Words per review")
ax[1].set_title("Review length by sentiment")
save_fig("nlp_01_class_and_length")
R["wc_median"] = df.groupby("sentiment")["word_count"].median()
R["wc_median"]

## 4. Train/test split and the modeling pipeline
- Stratified 80/20 split (`random_state=42`).
- TF-IDF is part of the **pipeline**, so the vocabulary and weights are learned from training text only. This avoids data leakage.
- TF-IDF settings: English stop words removed, unigrams and bigrams, at most 5,000 features, a term must appear in at least 2 reviews.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"], test_size=0.2, random_state=RANDOM_STATE, stratify=df["sentiment"])
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
R.update(n_train=len(X_train), n_test=len(X_test))


def make_pipe(model):
    return Pipeline([("tfidf", TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 2), min_df=2)),
                     ("clf", model)])


models = {
    "Multinomial Naive Bayes": make_pipe(MultinomialNB()),
    "Logistic Regression": make_pipe(LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipe(RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
}

## 5. Model comparison
Each model gets 5-fold cross-validation on the training set (a reliable estimate that does not touch the test set) and is then scored once on the test set.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows, fitted, preds = [], {}, {}
for name, pipe in models.items():
    cv_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    rows.append({"Model": name, "CV accuracy": cv_acc.mean(), "CV std": cv_acc.std(),
                 "Test accuracy": accuracy_score(y_test, pred), "Macro precision": p, "Macro recall": r, "Macro F1": f})
    fitted[name], preds[name] = pipe, pred
results = pd.DataFrame(rows).sort_values("CV accuracy", ascending=False).reset_index(drop=True)
best_name = results.loc[0, "Model"]
R.update(results=results, best_name=best_name)
results.round(3)

In [ ]:
order = list(results["Model"])
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(x - 0.2, results["CV accuracy"], 0.4, yerr=results["CV std"], capsize=3, color=BLUE, label="Cross-validated (train)")
ax.bar(x + 0.2, results["Test accuracy"], 0.4, color=RED, label="Test set")
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=12)
ax.set_ylim(max(0, results[["CV accuracy", "Test accuracy"]].min().min() - 0.1), 1.02)
ax.set_xlabel("Model")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by model")
ax.legend(loc="lower right")
save_fig("nlp_02_model_comparison")

## 6. Detailed evaluation of the best model
The best model is the one with the highest cross-validated accuracy.

In [ ]:
best_pred = preds[best_name]
report_text = classification_report(y_test, best_pred, zero_division=0)
cm = confusion_matrix(y_test, best_pred, labels=classes)
R.update(report_text=report_text, cm=cm, classes=classes)
print("Best model:", best_name)
print(report_text)

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=14, color="white" if v > cm.max() / 2 else "black")
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticks(range(len(classes)))
ax.set_yticklabels(classes)
ax.set_xlabel("Predicted sentiment")
ax.set_ylabel("Actual sentiment")
ax.set_title(f"Test confusion matrix: {best_name}")
save_fig("nlp_03_confusion_matrix")

### Error analysis
Looking at reviews the model got wrong shows what kind of text it finds hard.

In [ ]:
wrong = pd.DataFrame({"review": X_test.values, "actual": y_test.values, "predicted": best_pred})
wrong = wrong[wrong["actual"] != wrong["predicted"]]
R["n_wrong"] = len(wrong)
R["wrong_examples"] = wrong.sample(min(3, len(wrong)), random_state=RANDOM_STATE) if len(wrong) else wrong
print(f"{len(wrong)} of {len(X_test)} test reviews misclassified")
for _, r in R["wrong_examples"].iterrows():
    print(f"- actual {r['actual']}, predicted {r['predicted']}: {r['review'][:160]}...")

## 7. Which words drive the predictions?
For the Logistic Regression model, each word (or word pair) has a weight per class. The largest weights are the phrases that most strongly signal a class.

In [ ]:
lr = fitted["Logistic Regression"]
terms = np.array(lr.named_steps["tfidf"].get_feature_names_out())
coef = lr.named_steps["clf"].coef_
lr_classes = list(lr.named_steps["clf"].classes_)
if coef.shape[0] == 1:                      # binary problem: one weight vector for the second class
    class_weights = {lr_classes[1]: coef[0], lr_classes[0]: -coef[0]}
else:
    class_weights = {c: coef[i] for i, c in enumerate(lr_classes)}
top_terms = {c: pd.Series(w, index=terms).sort_values(ascending=False).head(10) for c, w in class_weights.items()}
R["top_terms"] = top_terms

fig, ax = plt.subplots(1, len(top_terms), figsize=(5.2 * len(top_terms), 5), squeeze=False)
for a, (c, s) in zip(ax[0], top_terms.items()):
    a.barh(s.index[::-1], s.values[::-1], color=palette.get(c, BLUE))
    a.set_xlabel("Model weight (higher = stronger signal)")
    a.set_ylabel("Word or phrase")
    a.set_title(f"Top phrases for {c}")
save_fig("nlp_04_top_phrases")
pd.DataFrame({c: list(s.index) for c, s in top_terms.items()})

## 8. Write the project report
The next cell writes `reports/advanced_analysis_report.md` from the results above, so every number comes from this run.

In [ ]:
def pct(x):
    return f"{x:.1%}"


def md_table(frame, fmt="{:.3f}"):
    cols = list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in frame.iterrows():
        lines.append("| " + " | ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(lines)


best = results.iloc[0]
res_show = results.copy()
res_show["CV accuracy"] = [f"{m:.3f} +/- {s:.3f}" for m, s in zip(res_show["CV accuracy"], res_show["CV std"])]
res_show = res_show.drop(columns="CV std")
class_lines = "\n".join(f"- **{c}:** {n:,} reviews" for c, n in R["class_counts"].items())
terms_tbl = pd.DataFrame({"Class": list(top_terms), "Top phrases (strongest first)": [", ".join(f"*{t}*" for t in s.index[:8]) for s in top_terms.values()]})
gap = best["CV accuracy"] - best["Test accuracy"]
caution = ""
if meta["synthetic"]:
    caution += ("\n> **Important caution:** this run used the bundled synthetic dataset. Its reviews are built from a small set of template sentences, "
                "so the task is very easy and the scores say little about how the models would perform on real, messy text. "
                "Run the notebook with `USE_REAL_DATA = True` and internet access to evaluate on real reviews.\n")
if best["Test accuracy"] >= 0.99:
    caution += ("\n> **Near-perfect accuracy is a warning sign, not a success.** It usually means the data is too easy or repetitive. "
                "Duplicate texts were removed before splitting to prevent leakage.\n")
wrong_md = "\n".join(f"- *Actual {r['actual']}, predicted {r['predicted']}:* \"{r['review'][:160]}...\"" for _, r in R["wrong_examples"].iterrows()) or "- No misclassified test reviews."
domain_note = ("The reviews are about movies, not telecom services, so the learned phrases (for example about acting or plot) would not transfer directly to telecom feedback. "
               "The workflow, however, applies unchanged to any labelled review data."
               if not meta["synthetic"] else
               "The bundled data is synthetic, so conclusions about real customers cannot be drawn from it.")

report = f"""# Project Report: Advanced Data Analysis Using Natural Language Processing (NLP)

*This report is generated by `notebooks/nlp_sentiment_analysis.ipynb`; every number comes from that run.*
{caution}
## 1. Chosen technique: NLP with TF-IDF and supervised sentiment classification
Natural language processing lets a computer extract structured information from free text. Here, each review is converted into numbers with **TF-IDF** (term frequency-inverse document frequency): a word gets a high weight when it is frequent in a review but rare across all reviews, so distinctive words count more than common ones. We use single words and word pairs (**unigrams and bigrams**), so that phrases such as "not good" are captured. A supervised classifier then learns to predict the sentiment label from these numbers.

## 2. Dataset and application
- **Dataset:** {meta['name']}. {meta['desc']}
- **Size:** {R['n_before']:,} reviews loaded; {R['n_dup']:,} duplicate texts removed; **{R['n_kept']:,}** reviews used.
- **Classes:**
{class_lines}
- **Application:** companies receive thousands of reviews and support messages. Automatic sentiment analysis can flag dissatisfied customers early so retention teams can respond, and shows which phrases signal dissatisfaction.

## 3. Methodology
1. **Quality checks and de-duplication:** missing values dropped and duplicate texts removed **before** splitting, so no review appears in both the training and the test set.
2. **Text cleaning:** lowercase, HTML tags and punctuation removed, whitespace normalised.
3. **Split:** stratified 80/20 ({R['n_train']:,} training, {R['n_test']:,} test reviews, `random_state=42`).
4. **TF-IDF inside a pipeline:** English stop words removed, unigrams and bigrams, at most 5,000 features, minimum 2 reviews per term. The vocabulary is learned from training text only, which prevents data leakage.
5. **Models:** Multinomial Naive Bayes, Logistic Regression and Random Forest.
6. **Evaluation:** 5-fold stratified cross-validation on the training set, then a single evaluation on the test set. Accuracy plus macro-averaged precision, recall and F1.

## 4. Results
### 4.1 Model comparison
{md_table(res_show)}

![Model comparison](../images/nlp_02_model_comparison.png)

The best model by cross-validated accuracy is **{best_name}**: {pct(best['CV accuracy'])} in cross-validation and **{pct(best['Test accuracy'])} on the test set**. The gap between the two is {abs(gap) * 100:.1f} percentage points, so the cross-validated estimate {"was close to" if abs(gap) < 0.02 else "differs noticeably from"} the test result.

### 4.2 Best model in detail
```
{R['report_text']}
```
![Confusion matrix](../images/nlp_03_confusion_matrix.png)

{R['n_wrong']} of {R['n_test']:,} test reviews were misclassified. Examples:

{wrong_md}

### 4.3 Which phrases drive the predictions (Logistic Regression weights)
{md_table(terms_tbl)}

![Top phrases](../images/nlp_04_top_phrases.png)

## 5. Insights
- The best model reads the sentiment of {pct(best['Test accuracy'])} of unseen reviews correctly, compared with {pct(R['class_counts'].max() / R['class_counts'].sum())} for always guessing the most common class.
- The phrases above are the clearest signals of each sentiment. A business could track how often they appear to monitor satisfaction over time.
- Automatic classification can triage a large stream of reviews so people read the unhappy ones first. Misclassified examples show where a human check is still needed.

## 6. Limitations
- {domain_note}
- TF-IDF treats text as a bag of words and cannot fully handle sarcasm, negation spread over several words or context; transformer-based models handle these better.
- Only {R['n_test']:,} reviews are in the test set, so accuracy differences of about one percentage point between models are not meaningful.
- Phrase weights show association with a label, not the reason a customer was unhappy.

## 7. How to reproduce
```bash
pip install -r requirements.txt
jupyter notebook notebooks/nlp_sentiment_analysis.ipynb   # Run All
```
In Google Colab: upload the notebook and choose **Runtime > Run all**. The run writes this report to `reports/advanced_analysis_report.md` and the charts to `images/`.
"""
(REPORT_DIR / "advanced_analysis_report.md").write_text(report, encoding="utf-8")
print("Saved report to", REPORT_DIR / "advanced_analysis_report.md")